<style>
    img {
        display: block;
        margin-left: auto;
        margin-right: auto;
    }
</style>

<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Modulo 3: Algoritmos de aprendizaje por refuerzo: Q-Learning y SARSA</h1>
    <h3>Aprendizaje Automático Avanzado 2026</h3>
</div>

## Objetivo del notebook

En este notebook construiremos y ejecutaremos un entorno sencillo de aprendizaje por refuerzo para comparar **Q-Learning** y **SARSA**.

El agente se moverÃ¡ por una cuadrÃ­cula desde el estado inicial $[0,0]$ hasta el estado terminal $[3,3]$. En cada paso podrÃ¡ elegir una de cuatro acciones:

- **Arriba**: $[-1, 0]$
- **Abajo**: $[1, 0]$
- **Izquierda**: $[0, -1]$
- **Derecha**: $[0, 1]$

El objetivo es aprender una ruta con una recompensa acumulada alta. Como cada acciÃ³n tiene una penalizaciÃ³n de $-1$, el agente debe evitar tanto las casillas peligrosas como los recorridos innecesariamente largos.

La casilla $[3,2]$ tiene una recompensa de $-100$ y la casilla terminal $[3,3]$ tiene una recompensa de $100$. La siguiente imagen muestra la estructura del problema:

<img src="./Figures/007_RL.png" alt="CuadrÃ­cula del entorno de aprendizaje por refuerzo" style="width: 300px;"/>

El notebook sigue este recorrido:

1. Definir el entorno y sus reglas.
2. Crear un agente base con selecciÃ³n de acciones y Q-Table.
3. Ejecutar una polÃ­tica de referencia con acciones aleatorias.
4. Implementar y evaluar Q-Learning.
5. Implementar y evaluar SARSA.
6. Comparar el efecto del factor de descuento en ambos algoritmos.

## 1. Definición del entorno

La clase `Environment` representa la cuadrícula en la que se mueve el agente. Se encarga de aplicar las acciones, limitar los movimientos al tablero, calcular las recompensas, detectar el estado terminal y registrar el recorrido de cada episodio.

In [ ]:
# importamos las librerias necesarias
import pandas as pd
import numpy as np

# Formato de los decimales en Pandas y la semilla del Random
pd.options.display.float_format = '{:,.2f}'.format
np.random.seed(23)


class Environment(object):
    def __init__(self, action_penalty=-1.0):
        """
        Clase que representa y controla en entorno
        :param step_penalty:    Factor de descuento del Reward por acciÃ³n tomada
        """
        self.actions = None
        self.rewards = None
        self.action_penalty = None         # Penalización por cada paso dado
        self.state = None                  # Estado en el que se encuentra el agente
        self.final_state = None            # Estado final del entorno. Cuando el agente llega, se termina el episodio
        self.total_reward = None           # Contador de recompensas en el episodio
        self.actions_done = None           # Lista en la que se guardan los pasos (acciones) realizadas en cada episodio

    def reset(self):
        """
        Método que reinicia las variables del entorno y devuelve es estado inicial
        :return:    state
        """
        self.total_reward = None   # Inicializamos Reward a 0
        self.state = None        # Posicionamos al agente en el estado inicial
        self.actions_done = None     # Inicializamos la listas de pasos (acciones)
        return None
        
    def __apply_action(self, action):
        """
        Método que calcula el nuevo estado a partir de la acción a ejecutar
        :param action:    AcciÃ³n a ejecutar
        """
        self.state[0] += None
        self.state[1] += None

        # Si nos salimos del tablero por arriba o por abajo, nos quedamos en la posicion que estabamos
        if self.state[0] < 0:
            self.state[0] = None
        elif self.state[0] > len(self.rewards) - 1:
            self.state[0] -= None

        # Si nos salimos del tablero por los lados, nos quedamos en la posicion que estabamos
        if self.state[1] < 0:
            self.state[1] = None
        elif self.state[1] > len(self.rewards[0]) - 1:
            self.state[1] -= None
            
    def step(self, action):
        """
        Método que ejecuta una acción determinada del conjunto de acciones {Arriba, Abajo, Izquierda, Derecha}
        para guiar al agente en el entorno.
        :param action:    Acción a ejecutar
        :return:          (state, reward, is_final_state)
        """
        # Realizamos la acción (cambio de estado)
        # Guardamos el paso (accion) realizada
        # Comprobamos si hemos llegado al estado final
        # Calculamos el reward (recompensa) por la acciÃ³n tomada
        # Sumamos el reward (recompensa) total del episodio

        # Devolvemos es estado, el reward (recompensa) y si hemos llegado al estado final
        return None, None, None

    

    def print_path_episode(self):
        """
        Método que imprime por pantalla el camino seguido por el agente
        :return: 
        """
        path = [['-' for _ in range(len(self.rewards))] for _ in range(len(self.rewards[0]))]
        path[0][0] = '0'
        for index, step in enumerate(self.actions_done):
            path[step[0]][step[1]] = str(index + 1)

        print(pd.DataFrame(data=np.array([np.array(xi) for xi in path]),
                           index=["x{}".format(str(i)) for i in range(len(path))],
                           columns=["y{}".format(str(i)) for i in range(len(path[0]))]))


## 2. Agente base y política de referencia

Antes de implementar Q-Learning y SARSA, definiremos una clase base llamada `Learner`. Esta clase contiene la estructura común de los agentes:

- Una **Q-Table** con un valor para cada combinaciónn de estado y acción.
- Los hiperparámetros de aprendizaje, descuento y exploración.
- La lógica para seleccionar una acción.
- Métodos para mostrar la tabla, los mejores valores y las mejores acciones.

La clase base utilizará una política de referencia. Su método `update` no modifica la Q-Table, por lo que permite observar qué ocurre cuando el agente no aprende y solo selecciona acciones segúnn la información disponible.

La selección de acciones combina dos comportamientos:

- **Exploración**: elegir una acción al azar con probabilidad `ratio_exploration`.
- **Explotación**: elegir una de las acciones con mayor valor estimado en la Q-Table.

Como todos los valores comienzan en cero, al principio las acciones de explotación son equivalentes y se desempatan aleatoriamente.

In [ ]:
class Learner(object):

    def __init__(self, environment, learning_rate=0.1, discount_factor=0.1, ratio_exploration=0.05):
        """
        Clase que implementa un algoritmo de aprendiza por refuerzo
        Esta clase implementa un algoritmo de selección aleatoria de acciones
        :param environment:         Entorno en el que tomar las acciones
        :param learning_rate:       Factor de aprendizaje
        :param discount_factor:     Factor de descuento (0=Estrategia a corto plazo, 1=Estrategia a largo plazo)
        :param ratio_exploration:   Ratio de exploraciÃ³n
        """
        self.environment = None
        self.q_table = [[[0.0 for _ in self.environment.actions]
                         for _ in range(len(self.environment.rewards))]
                        for _ in range(len(self.environment.rewards[0]))]
        
        self.learning_rate = None
        self.discount_factor = None
        self.ratio_exploration = None

    @property
    def name(self):
        return 'random'

    def get_next_action(self, state):
        """
        Método que selecciona la siguiente acciÃ³n a tomar:
            Aleatoria -> si el ratio de exploraciÃ³n es inferior al umbral
            Mejor AcciÃ³n -> si el ratio de exploraciÃ³n es superior al umbral
        :param state:   Estado del agente
        :return:        next_action
        """

        if np.random.uniform() < self.ratio_exploration:
            # Seleccionamos una opción al azar
            next_action = None
        else:
            # Seleccionamos la acciÃ³n que nos de mayor valor. Si hay empate, Seleccionamos una al azar
            idx_action = np.random.choice(np.flatnonzero(
                self.q_table[state[0]][state[1]] == np.array(self.q_table[state[0]][state[1]]).max()
            ))
            next_action = None

        return next_action

    def update(self, **kwargs):
        """
        Actualiza la Q-Table
        :param kwargs: 
        """
        pass

    def print_q_table(self):
        """
        Método que imprime por pantalla la Q-Table
        """
        q_table = []
        for x, row in enumerate(self.q_table):
            for y, col in enumerate(row):
                q = deepcopy(col)
                q.insert(0, 'x{},y{}'.format(x,y))
                q_table.append(q)
        print(pd.DataFrame(data=q_table,
                           columns=['Estado', 'Arriba', 'Abajo', 'Izquierda', 'Derecha'])
              .to_string(index=False))

    def print_best_actions_states(self):
        """
        Método que imprime por pantalla la mejor opciÃ³n a realizar en cada uno de los estados
        """

        best = [[list(self.environment.actions)[np.argmax(col)] for col in row] for row in self.q_table]
        print(pd.DataFrame(data=np.array([np.array(xi) for xi in best]),
                           index=["x{}".format(str(i)) for i in range(len(best))],
                           columns=["y{}".format(str(i)) for i in range(len(best[0]))]))
        
    def print_best_values_states(self):
        """
        Método que imprime por pantalla el valor de la mejor opciÃ³n a realizar en cada uno de los estados
        """
        best = [[max(vi) for vi in row] for row in self.q_table]
        print(pd.DataFrame(data=np.array([np.array(xi) for xi in best]),
                           index=["x{}".format(str(i)) for i in range(len(best))],
                           columns=["y{}".format(str(i)) for i in range(len(best[0]))]))


## 3. Ejecución del agente en el entorno

La función `run_agent` coordina la interacción entre el entorno y el agente durante varios episodios.

En cada episodio:

1. El entorno se reinicia en el estado inicial `[0,0]`.
2. El agente selecciona una acción.
3. El entorno devuelve el nuevo estado, la recompensa y si se alcanzÃ³ el estado terminal.
4. El agente actualiza su Q-Table cuando el algoritmo lo permite.
5. Se almacenan el número de pasos y la recompensa acumulada.

La función `print_process_info` resume los resultados del mejor episodio registrado: muestra la Q-Table, los mejores valores por estado, las acciones preferidas y el camino recorrido. Estas salidas permiten observar no solo la recompensa final, sino también qué aprendió el agente.

In [ ]:
from copy import deepcopy

def run_agent(learner=Learner, num_episodes=10, learning_rate=0.1, discount_factor=0.1, ratio_exploration=0.05,
              verbose=False):
    """
    Método que ejecuta el proceso de aprendizaje del agente en un entorno
    :param learner:              Algoritmo de Aprendizaje
    :param num_episodes:         Número de veces que se ejecuta (o aprende) el agente en el entorno
    :param learning_rate:        Factor de Aprendizaje
    :param discount_factor:      Factor de descuento (0=Estrategia a corto plazo, 1=Estrategia a largo plazo)
    :param ratio_exploration:    Ratio de exploraciÃ³n
    :param verbose:              Boolean, si queremos o no imprimir por pantalla informaciÃ³n del proceso
    :return:                     (episodes_list, best_episode)
    """

    # Instanciamos el entorno
    

    # Instanciamos el método de aprendizaje
    

    # Variables para guardar la informaciÃ³n de los episodios
    episodes_list = []
    best_reward = float('-inf')
    best_episode = None

    for n_episode in range(0, num_episodes):
        state = environment.reset()
        reward = None
        is_final_state = None
        num_steps_episode = 0
        while not is_final_state:
            old_state = state[:]
            # Accion a realizar; bien sea explotando la Q-Table o explorando
            
            # Realizamos la accion
            
            # Si usamos el SARSA realizamos una accion en el nuevo estado S_t+1
            
            # Actualizamos el entorno
            
            # Sumamos un paso al episocio

        # Guardamos la informaciÃ³n del episodio
        
        # Guardamos el mejor episodio
        if environment.total_reward >= best_reward:
            best_reward = None
            best_episode = {}

        if verbose:
            # Imprimimos la informaciÃ³n de los episodios
            print('EPISODIO {} - Numero de acciones: {} - Reward: {}'
                  .format(n_episode + 1, num_steps_episode, environment.total_reward))

    return None, None


def print_process_info(episodes_list, best_episode, print_best_episode_info=True,
                       print_q_table=True, print_best_values_states=True,
                       print_best_actions_states=True, print_steps=True, print_path=True):
    """
    MÃ©todo que imprime por pantalla los resultados de la ejecuciÃ³n
    """
    if print_best_episode_info:
        print('\nMEJOR (ÃšLTIMO) EPISODIO:\nEPISODIO {}\n\tNumero de acciones: {}\n\tReward: {}'
              .format(best_episode['num_episode'],
                      len(best_episode['episode'].actions_done),
                      best_episode['episode'].total_reward))

    if print_q_table:
        print('\nQ_TABLE:')
        best_episode['learner'].print_q_table()
        
    if print_best_values_states:
        print('\nBEST Q_TABLE VALUES:')
        best_episode['learner'].print_best_values_states()

    if print_best_actions_states:
        print('\nBEST ACTIONS:')
        best_episode['learner'].print_best_actions_states()

    if print_steps:
        print('\nPasos: \n   {}'.format(best_episode['episode'].actions_done))

    if print_path:
        print('\nPATH:')
        best_episode['episode'].print_path_episode()


### 3.1. Política de referencia sin aprendizaje

Primero ejecutaremos el agente base. Esta ejecución sirve como punto de comparación: la Q-Table permanece sin actualizarse, por lo que el agente no incorpora la experiencia de episodios anteriores.

La semilla aleatoria se fija al inicio del notebook para que los resultados sean reproducibles. Aun así, el recorrido puede ser largo porque el agente no ha aprendido una ruta eficiente.

In [ ]:
#Corremos el agente

#imprimimos el proceso


## 4. Q-Learning: implementación y ejecución

Q-Learning actualiza el valor de una pareja estado-acción utilizando la mejor acción posible en el estado siguiente. Su regla de actualización es:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma\max_{a'}Q(s',a') - Q(s,a)\right]$$

En el código, `actual_q_value` representa la estimación actual y `future_max_q_value` representa el objetivo de aprendizaje. Cuando el agente llega al estado terminal, no se añade valor futuro: solo se utiliza la recompensa obtenida en ese paso.

El pseudocódigo resume el procedimiento:

<img src="./Figures/013_qlearning.png" alt="PseudocÃ³digo de Q-Learning" style="width: 600px;"/>

Después de definir la clase `QLearner`, se ejecutan dos experimentos. Ambos utilizan la misma tasa de aprendizaje y el mismo nivel de exploración, pero modifican el factor de descuento:

- **Estrategia a corto plazo**: `discount_factor=0.1`. Da más peso a las recompensas inmediatas.
- **Estrategia a largo plazo**: `discount_factor=0.9`. Da más importancia a las recompensas futuras.

La comparación debe centrarse en la recompensa acumulada, el número de pasos y la ruta que aprende el agente.

In [ ]:
class QLearner(Learner):

    @property
    def name(self):
        return 'QLearner'

    def update(self, environment, old_state, action_taken, reward_action_taken, new_state, is_final_state, **kwargs):
        """
        MÃ©todo que implementa el Algoritmo de aprendizaje del Q-Learning
        :param environment:           Entorno en el que tomar las acciones
        :param old_state:             Estado actual
        :param action_taken:          Acción a realizar
        :param reward_action_taken:   Recompensa obtenida por la acciÃ³n tomada
        :param new_state:             Nuevo estado al que se mueve el agente
        :param is_final_state:        Boolean. Devuelve True si el agente llega al estado final; si no, False
        :param kwargs: 
        """
        # Obtengo el identificador de la acciÃ³n
        

        # Obtengo el valor de la acciÃ³n tomada
        
        future_q_value_options = self.q_table[new_state[0]][new_state[1]]
        future_max_q_value = reward_action_taken + self.discount_factor * max(future_q_value_options)
        if is_final_state:
            future_max_q_value = reward_action_taken    # Reward mÃ¡ximo si llego a la posiciÃ³n final

        self.q_table[old_state[0]][old_state[1]][idx_action_taken] = \
            actual_q_value + self.learning_rate * (future_max_q_value - actual_q_value)


#### Estrategia a corto plazo

In [ ]:
episodes_list, best_episode = run_agent(learner=QLearner,
                                        num_episodes=25,
                                        learning_rate=0.1,
                                        discount_factor=0.1,
                                        ratio_exploration=0.05,
                                        verbose=True)

print_process_info(episodes_list=episodes_list,
                   best_episode=best_episode)


#### Estrategia a largo plazo

In [ ]:
episodes_list, best_episode = run_agent(learner=QLearner,
                                        num_episodes=25,
                                        learning_rate=0.1,
                                        discount_factor=0.9,
                                        ratio_exploration=0.05,
                                        verbose=True)

print_process_info(episodes_list=episodes_list,
                   best_episode=best_episode)

## 5. SARSA: implementación y ejecución

SARSA actualiza la Q-Table utilizando la acción que la política realmente seleccionará en el estado siguiente:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma Q(s',a') - Q(s,a)\right]$$

La diferencia con Q-Learning está en el valor futuro:

- **Q-Learning** utiliza $\max_{a'} Q(s',a')$, es decir, la mejor acción posible.
- **SARSA** utiliza $Q(s',a')$, donde $a'$ es la acción que el agente ha elegido para el siguiente estado.

Por esta razón, SARSA incorpora en su aprendizaje el comportamiento exploratorio que realmente seguirá el agente.

<img src="./Figures/014_sarsa.png" alt="PseudocÃ³digo de SARSA" style="width: 600px;"/>

Al igual que con Q-Learning, se evaluarán una configuración con mayor énfasis en recompensas inmediatas y otra con mayor énfasis en recompensas futuras.

In [ ]:
class SARSALearner(Learner):

    @property
    def name(self):
        return 'SARSA'

    def update(self, environment, old_state, action_taken, reward_action_taken, new_state, new_action, is_final_state):
        """
        MÃ©todo que implementa el algoritmo de aprendizaje SARSA
        :param environment:           Entorno en el que tomar las acciones
        :param old_state:             Estado actual
        :param action_taken:          AcciÃ³n a realizar
        :param reward_action_taken:   Recompensa obtenida por la acciÃ³n tomada
        :param new_state:             Nuevo estado al que se mueve el agente 
        :param new_action:            AcciÃ³n a tomar en el nuevo estado
        :param is_final_state:        Boolean. Devuelve True si el agente llega al estado final; si no, False 
        """
        # Obtengo el identificador de la acciÃ³n
        idx_action_taken = list(environment.actions).index(action_taken)

        # Obtengo el valor de la acciÃ³n tomada
        actual_q_value_options = self.q_table[old_state[0]][old_state[1]]
        actual_q_value = actual_q_value_options[idx_action_taken]

        future_q_value_options = self.q_table[new_state[0]][new_state[1]]

        idx_new_action_taken = list(environment.actions).index(new_action)
        future_new_action_q_value = \
            reward_action_taken + self.discount_factor * future_q_value_options[idx_new_action_taken]
        if is_final_state:
            # Reward mÃ¡ximo si llego a la posiciÃ³n final
            future_new_action_q_value = reward_action_taken

        self.q_table[old_state[0]][old_state[1]][idx_action_taken] = \
            actual_q_value + self.learning_rate * (future_new_action_q_value - actual_q_value)


#### Estrategia a corto plazo

In [ ]:
episodes_list, best_episode = run_agent(learner=SARSALearner,
                                        num_episodes=25,
                                        learning_rate=0.1,
                                        discount_factor=0.1,
                                        ratio_exploration=0.05,
                                        verbose=True)

print_process_info(episodes_list=episodes_list,
                   best_episode=best_episode)

#### Estrategia a largo plazo

In [ ]:
episodes_list, best_episode = run_agent(learner=SARSALearner,
                                        num_episodes=25,
                                        learning_rate=0.1,
                                        discount_factor=0.9,
                                        ratio_exploration=0.05,
                                        verbose=True)

print_process_info(episodes_list=episodes_list,
                   best_episode=best_episode)

## 6. Comparación de estrategias

En la última parte estudiaremos cómo cambia el aprendizaje cuando modificamos el factor de descuento $\gamma$. Se probarán los valores `0.01`, `0.3`, `0.7` y `0.99` para Q-Learning y SARSA.

Las gráficas muestran dos perspectivas complementarias:

- **Recompensa por episodio**: permite observar la variabilidad y el comportamiento de cada episodio.
- **Recompensa promedio acumulada**: permite observar la tendencia general del aprendizaje.

Al interpretar los resultados, conviene comparar las curvas considerando que el entorno penaliza cada paso. Una ruta con menos movimientos suele producir una recompensa mayor, siempre que evite la casilla con penalización de `-100`.

### 6.1. Q-Learning

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline

discount_factors = [0.01, 0.3, 0.7, 0.99]
experiments = []
for disfac in discount_factors:
    episodes_list, best_episode = run_agent(learner=QLearner,
                                            num_episodes=30,
                                            learning_rate=0.1,
                                            discount_factor=disfac,
                                            ratio_exploration=0.05)
    experiments.append({'Discount Factor': str(disfac),
                        'Rewards': [episode[2] for episode in episodes_list]})

plt.figure(figsize=(20, 8))
for experiment in experiments:
    plt.subplot(1, 2, 1)
    plt.title('EvoluciÃ³n Reward por Episodios VS Discount Factor')
    plt.plot(experiment['Rewards'], label=experiment['Discount Factor'])
    plt.legend(loc='lower right')

    plt.subplot(1, 2, 2)
    plt.title('EvoluciÃ³n Acumulada Reward por Episodios VS Discount Factor')
    acum_avg_reward = (np.cumsum(experiment['Rewards']) / (np.arange(len(experiment['Rewards'])) + 1))[0:]
    plt.plot(acum_avg_reward, label=experiment['Discount Factor'])
    plt.legend(loc='lower right')

plt.show()


### 6.2. SARSA

El siguiente experimento repite la comparación de factores de descuento utilizando SARSA. Al finalizar, compara sus curvas con las de Q-Learning y observa cómo influye el hecho de que SARSA actualice los valores usando la acción que la política realmente selecciona.

In [ ]:
discount_factors = [0.01, 0.3, 0.7, 0.99]
experiments = []
for disfac in discount_factors:
    episodes_list, best_episode = run_agent(learner=SARSALearner,
                                            num_episodes=30,
                                            learning_rate=0.1,
                                            discount_factor=disfac,
                                            ratio_exploration=0.05)
    experiments.append({'Discount Factor': str(disfac),
                        'Rewards': [episode[2] for episode in episodes_list]})

plt.figure(figsize=(20, 8))
for experiment in experiments:
    plt.subplot(1, 2, 1)
    plt.title('EvoluciÃ³n Reward por Episodios VS Discount Factor')
    plt.plot(experiment['Rewards'], label=experiment['Discount Factor'])
    plt.legend(loc='lower right')

    plt.subplot(1, 2, 2)
    plt.title('EvoluciÃ³n Acumulada Reward por Episodios VS Discount Factor')
    acum_avg_reward = (np.cumsum(experiment['Rewards']) / (np.arange(len(experiment['Rewards'])) + 1))[0:]
    plt.plot(acum_avg_reward, label=experiment['Discount Factor'])
    plt.legend(loc='lower right')

plt.show()